# 🇺🇸 A Century of American Carbon
### A data story, built with the `viz_lib` library

This notebook presents the story one graph at a time. **Run each cell from top to bottom** (`Shift+Enter`). Every chart is produced by a function from the `viz_lib` library — the same small, importable package described in the project.

**Story arc:** global context → the United States by fuel → beyond CO₂ → the historical bill → the full poster.

## Setup
Run this **once**. It fetches the library + demo data and imports everything.

> If the repository is private, either make it public for the demo, or add a GitHub token to the clone URL.

In [ ]:
# --- Fetch the library + demo data, then import viz_lib ---
import os, sys
REPO = 'visualisation-lib'
if not os.path.exists(REPO):
    !git clone -q -b claude/viz-lib-plot-requirements-dbjbc0 https://github.com/rinikhaneja/visualisation-lib.git
!pip -q install matplotlib pandas

sys.path.insert(0, os.path.join(REPO, 'src'))        # the library
sys.path.insert(0, os.path.join(REPO, 'reports'))    # the poster module
DATA = os.path.join(REPO, 'data', 'demo')            # demo CSVs

import pandas as pd
import matplotlib.pyplot as plt
from viz_lib import ranked_bar, stacked_area
from viz_lib.theme import apply_theme, series_color
apply_theme()
print('viz_lib ready ✓')

## 1 · Global context — who emits the most CO₂ per person?
Two peer groups, drawn on the **same color scale** so they can be compared at a glance. ▲/▼ show the change since 2014.

In [ ]:
# Load the per-capita snapshot (2014 & 2024) and reshape to one row per country
df = pd.read_csv(f'{DATA}/co2_per_capita.csv')
df = df.pivot_table(index='Entity', columns='Year', values='CO₂ emissions per capita')
df.columns = [f'y{c}' for c in df.columns]
df = df.reset_index()

OIL  = ['Qatar','Kuwait','Brunei','Bahrain','Trinidad and Tobago',
        'Saudi Arabia','United Arab Emirates','Oman']
ECON = ['United States','Russia','North America','China',
        'European Union (27)','World','United Kingdom','India']
vmax  = df[df.Entity.isin(OIL + ECON)].y2024.max()   # shared color scale
world = df.loc[df.Entity == 'World', 'y2024'].iloc[0]
pick  = lambda names: df[df.Entity.isin(names)]

ranked_bar(pick(OIL), category='Entity', value='y2024', compare='y2014',
           vmax=vmax, unit='t', reference=world, reference_label='World average',
           title='A few small, oil-rich nations emit the most CO₂ per person',
           subtitle='Tonnes of CO₂ per person, 2024')
plt.show()

In [ ]:
ranked_bar(pick(ECON), category='Entity', value='y2024', compare='y2014',
           vmax=vmax, unit='t',
           title='Among big economies, the US still emits the most per person',
           subtitle='Tonnes of CO₂ per person, 2024 — same scale as the oil producers')
plt.show()

## 2 · Zooming into the US — CO₂ by fuel (the hero)
A century-long stacked area: **coal → oil → gas**, annotated with the events that shaped it.

In [ ]:
h = pd.read_csv(f'{DATA}/us_co2_by_fuel.csv')
FUELS = ['Coal','Oil','Gas','Cement','Flaring','Other industry']
for f in FUELS:
    h[f] = pd.to_numeric(h[f], errors='coerce') / 1e9   # tonnes -> billion tonnes

EVENTS = [{'year':1932,'label':'1932\nGreat Depression','y':0.42},
          {'year':1945,'label':'1945\nWWII','y':0.72},
          {'year':1973,'label':'1973\nOil shock','y':0.9},
          {'year':2007,'label':'2007\nemissions peak','y':0.98},
          {'year':2020,'label':'2020\nCOVID','y':0.62}]

stacked_area(h, x='Year', series=FUELS, y_label='Billion tonnes CO₂ / year',
             title='Coal gave way to oil and gas',
             subtitle='US CO₂ emissions by fuel or industry, 1800–2024',
             events=EVENTS)
plt.show()

## 3 · Beyond CO₂ — per-capita greenhouse gases
Total greenhouse gases (CO₂ **+ methane + nitrous oxide**), in CO₂-equivalents. The US has roughly halved its per-person footprint — but it is still ~18 t/person.

In [ ]:
g = pd.read_csv(f'{DATA}/us_percapita_ghg.csv')
col = [c for c in g.columns if 'greenhouse' in c.lower()][0]
x, y = g.Year.to_numpy(), g[col].to_numpy()

fig, ax = plt.subplots(figsize=(9, 4.5))
c = series_color(0)
ax.plot(x, y, color=c, linewidth=2.2, solid_capstyle='round')
ax.fill_between(x, y, color=c, alpha=0.08)
ax.set_ylim(0, y.max()*1.08); ax.set_xlim(x.min(), x.max())
ax.annotate(f'{y[-1]:.0f} t', xy=(x[-1], y[-1]), xytext=(6, 0),
            textcoords='offset points', va='center', fontweight='bold', color=c)
for s in ('top','right'): ax.spines[s].set_visible(False)
ax.tick_params(length=0)
ax.set_title('Even as CO₂ fell, the US still emits ~18 t per person',
             loc='left', fontsize=14, fontweight='bold')
plt.show()

## 4 · The historical bill — US share of cumulative CO₂
Of **all** the CO₂ the world has ever emitted from each source, how much came from the United States?

In [ ]:
rows = []
for fuel, name in [('Oil','share_cumulative_oil.csv'),
                   ('Coal','share_cumulative_coal.csv'),
                   ('Cement','share_cumulative_cement.csv')]:
    d = pd.read_csv(f'{DATA}/{name}')
    col = [c for c in d.columns if 'Share' in c][0]
    val = d[(d.Entity == 'United States') & (d.Year == 2024)][col].iloc[0]
    rows.append({'Fuel': fuel, 'share': val})

ranked_bar(pd.DataFrame(rows), category='Fuel', value='share', value_fmt='{:.0f}%',
           title='A quarter of all oil CO₂ ever, from one country',
           subtitle="US share of the world's cumulative CO₂, by source")
plt.show()

## 5 · The full poster
Everything above, composed onto one print-ready canvas by `reports/us_carbon_poster.py`.

In [ ]:
import us_carbon_poster as poster
fig = poster.build_poster(DATA)   # reads the demo CSVs, returns the Figure
fig

---
*Built with `viz_lib` — a small, importable plotting library.* 
Data: Global Carbon Budget (2025) via Our World in Data (CC BY).